# YouTube Chatbot — HuggingFace (no paid API)


In [2]:
# ── Install dependencies ──────────────────────────────────────────────────
!pip install -q youtube-transcript-api langchain langchain-community \
                 langchain-huggingface langchain-core faiss-cpu \
                 sentence-transformers huggingface-hub

In [3]:
import os
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFaceEndpoint
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

ModuleNotFoundError: No module named 'langchain.text_splitter'

In [ ]:
# ── CONFIG — replace with your HuggingFace token ─────────────────────────
HF_TOKEN    = "your_huggingface_token_here"
VIDEO_ID    = "Gfr50f6ZBvo"

EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
LLM_REPO_ID = "mistralai/Mistral-7B-Instruct-v0.3"

In [ ]:
# ── 1. Fetch transcript ───────────────────────────────────────────────────
try:
    transcript_list = YouTubeTranscriptApi.get_transcript(VIDEO_ID, languages=["en"])
    transcript = " ".join(chunk["text"] for chunk in transcript_list)
    print(transcript[:500])
except TranscriptsDisabled:
    raise RuntimeError("No captions available for this video.")

In [ ]:
# ── 2. Split into chunks ──────────────────────────────────────────────────
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks   = splitter.create_documents([transcript])
print(f"Total chunks: {len(chunks)}")

In [ ]:
# ── 3. Embeddings + Vector Store ──────────────────────────────────────────
embeddings   = HuggingFaceEmbeddings(model_name=EMBED_MODEL)
vector_store = FAISS.from_documents(chunks, embeddings)
print("Vector store ready.")

In [ ]:
# ── 4. Retriever ──────────────────────────────────────────────────────────
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)
retriever.invoke("What is deepmind")

In [ ]:
# ── 5. LLM via HuggingFace Inference API (no download) ───────────────────
llm = HuggingFaceEndpoint(
    repo_id=LLM_REPO_ID,
    huggingfacehub_api_token=HF_TOKEN,
    temperature=0.2,
    max_new_tokens=512,
)

In [ ]:
# ── 6. Prompt ─────────────────────────────────────────────────────────────
prompt = PromptTemplate(
    template="""
You are a helpful assistant.
Answer ONLY from the provided transcript context.
If the context is insufficient, just say you don't know.

{context}
Question: {question}
""",
    input_variables=["context", "question"],
)

In [ ]:
# ── 7. Manual retrieval test ──────────────────────────────────────────────
def format_docs(retrieved_docs):
    return "\n\n".join(doc.page_content for doc in retrieved_docs)

question       = "is the topic of nuclear fusion discussed in this video? if yes then what was discussed"
retrieved_docs = retriever.invoke(question)
context_text   = format_docs(retrieved_docs)

final_prompt = prompt.invoke({"context": context_text, "question": question})
answer = llm.invoke(final_prompt.text)
print(answer)

In [ ]:
# ── 8. Full LCEL chain ────────────────────────────────────────────────────
parallel_chain = RunnableParallel({
    "context":  retriever | RunnableLambda(format_docs),
    "question": RunnablePassthrough(),
})

parser     = StrOutputParser()
main_chain = parallel_chain | prompt | llm | parser

print(parallel_chain.invoke("who is Demis"))

In [ ]:
# ── 9. Ask anything ───────────────────────────────────────────────────────
print(main_chain.invoke("Can you summarize the video"))